# Dataset

## 7000 invoice images with JSON

This dataset contains 7000 invoice images and their corresponding JSON files. There are 7 types of invoices in this dataset, each one containing 1000 examples each. The data content in the invoices has been generated using Python Faker.

Dataset URL: https://www.kaggle.com/datasets/ananthakrishnanpv/7000-invoice-images-with-json

In [1]:
import json

import pandas as pd
import requests
from sympy.strategies import tryit
from tqdm.auto import tqdm
from rapidfuzz import fuzz
import re

import env

HEADERS = {
    "X-API-KEY": env.DOXTRACT_API_KEY,
    "X-API-SECRET": env.DOXTRACT_API_SECRET
}

def doxtract(data, files):
    """Test verification endpoint"""
    url = f"{env.DOXTRACT_API_URL}/api/read"

    response = requests.post(url, data=data, headers=HEADERS, files=files)

    return response.json() if response.ok else None

/home/sadid/anaconda3/envs/python-3-13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
response = requests.get(f"{env.DOXTRACT_API_URL}/api/templates", headers=HEADERS)
templates = response.json() if response.ok else None
print(json.dumps(templates, indent=2))

{
  "total": 9,
  "skip": 0,
  "limit": 10,
  "data": [
    {
      "id": "6a30f57e4a51fc486f3e24f7",
      "type": "invoice_template_07",
      "description": "",
      "model": "freemium",
      "created": "2026-06-16T09:38:35.019000+00:00"
    },
    {
      "id": "6a30f6ae4a51fc486f3e24fa",
      "type": "invoice_template_04",
      "description": "",
      "model": "freemium",
      "created": "2026-06-16T09:27:13.370000+00:00"
    },
    {
      "id": "6a30f2574a51fc486f3e24df",
      "type": "invoice_template_02",
      "description": "",
      "model": "freemium",
      "created": "2026-06-16T07:17:59.183000+00:00"
    },
    {
      "id": "6a30f5074a51fc486f3e24f4",
      "type": "invoice_template_06",
      "description": "",
      "model": "freemium",
      "created": "2026-06-16T07:02:31.707000+00:00"
    },
    {
      "id": "6a30f3fb4a51fc486f3e24ed",
      "type": "invoice_template_05",
      "description": "",
      "model": "freemium",
      "created": "2026-06-16T06:5

In [3]:
template_dict = {
    item["type"]: item["id"]
    for item in templates["data"]
    if "invoice_template_" in item.get("type", "")
}

print(template_dict)

{'invoice_template_07': '6a30f57e4a51fc486f3e24f7', 'invoice_template_04': '6a30f6ae4a51fc486f3e24fa', 'invoice_template_02': '6a30f2574a51fc486f3e24df', 'invoice_template_06': '6a30f5074a51fc486f3e24f4', 'invoice_template_05': '6a30f3fb4a51fc486f3e24ed', 'invoice_template_03': '6a30f34b4a51fc486f3e24e6', 'invoice_template_01': '6a30cb20861f1b174a39c805'}


In [4]:
def parse_amount(text_string: str) -> float | None:
    """Extracts a numeric financial amount from a text string."""
    if not isinstance(text_string, str):
        return None

    # Matches digits with optional commas and an optional decimal point
    # It also handles optional negative signs (-£534.11)
    match = re.search(r"(-?\d+[\d,]*\.\d+|-?\d+)", text_string)

    if match:
        # Strip commas used for thousands separators (e.g., 1,534.11 -> 1534.11)
        clean_num = match.group(1).replace(",", "")
        return float(clean_num)

    return None

In [5]:
data_root = "data/7k"

def safe_dict_get(d, key):
    if not isinstance(d, dict):
        return {}
    return d.get(key, {}) if isinstance(d.get(key), dict) else d.get(key)

def safe_str(x):
    return "" if x is None else str(x)

def safe_float(x):
    return None if x is None else float(x)

def print_details(file_name):
    with open(data_root + f"/json/{file_name}.json") as f:
        truth = json.load(f)

        try:
            contact = "\n".join(truth["buyer"].values())
            invoice_no  = str(truth["invoice"]["number"])

            total = 0
            if "payment" in truth.keys() and "total" in truth["payment"].keys():
                total = truth["payment"]["total"]
            elif "payment" in truth.keys() and "account_number" in truth["payment"].keys():
                total = "\n".join([str(x["unit_price"]) for x in truth["products"]])
            elif "tax" in truth.keys() and "amount_0" in truth["tax"].keys():
                total = truth["tax"]["amount_0"]
            elif "tax" in truth.keys() and "amount_excluding_tax" in truth["tax"].keys():
                total = truth["tax"]["amount_excluding_tax"]

            return dict(invoice_no=invoice_no, total=total, contact=contact)
        except Exception as e:
            print(f"Error reading {file_name}: {e}")
            return {"invoice_no": "", "total": 0, "contact": ""}

def process_row(file_name, template_name):
    result = doxtract({
        "data": json.dumps({"templates": [template_dict[template_name]]})
    }, {
        "files": open(f"{data_root}/image/{file_name}.png", "rb")
    })

    tru = print_details(file_name)
    contact_tru = tru["contact"]
    total_tru = tru["total"]
    invoice_tru = tru["invoice_no"]

    if result is not None:
        contact_hat = safe_str(result.get("result", {}).get("contact"))
        total_hat = safe_str(result.get("result", {}).get("total"))
        invoice_hat = safe_str(result.get("result", {}).get("invoice_no"))
    else:
        contact_hat = None
        total_hat = None
        invoice_hat = None

    row = {
        "template": template_name,
        "filename": file_name,
        "contact_tru": contact_tru,
        "contact_hat": contact_hat,
        "total_tru": total_tru,
        "total_hat": total_hat,
        "invoice_tru": invoice_tru,
        "invoice_hat": invoice_hat
    }
    return row, result is None

def generate_results(template_index, test_rows=999):
    rows = []
    errors = 0
    progress = tqdm(total=1000, leave=False)
    template_name = f"invoice_template_0{template_index+1}"
    progress.set_postfix({"template": template_name})
    for i in range(test_rows):
        if template_index == 0:
            file_name = str(i+1).zfill(3)
        else:
            file_name = str(template_index) + str(i+1).zfill(4)

        row, error = process_row(file_name, template_name)
        rows.append(row)
        errors += error
        progress.update()
        progress.set_postfix({"template": template_name, "error": error})

    return pd.DataFrame(rows)

In [6]:

def parse_amount(x):
    try:
        import re
        nums = re.findall(r'-?\d+(?:\.\d+)?', str(x))
        return float(nums[0]) if nums else None
    except:
        return None

def calculate_metrics(results):
    df = results.copy()

    df["invoice_matched"] = (
        df["invoice_tru"].astype(str)
        == df["invoice_hat"].astype(str)
    )

    df["total_hat_amount"] = df["total_hat"].apply(parse_amount)

    df["total_matched"] = (
        df["total_tru"].astype(float)
        == df["total_hat"].apply(parse_amount)
    )

    df["contact_score"] = df.apply(
        lambda r: fuzz.token_sort_ratio(
            str(r["contact_tru"]),
            str(r["contact_hat"])
        ),
        axis=1,
    )

    df["contact_matched"] = df["contact_score"] >= 90

    df["document_matched"] = (
        df["invoice_matched"]
        & df["total_matched"]
        & df["contact_matched"]
    )

    print("Accuracies -")
    print("invoice_no :", df["invoice_matched"].mean() * 100)
    print("total_amount:", df["total_matched"].mean() * 100)
    print("contact:", df["contact_matched"].mean() * 100)
    print("Overall:", df["document_matched"].mean() * 100)
    return df

In [7]:
df0 = generate_results(0)
metric0 = calculate_metrics(df0)

metric0.head()

Accuracies -
invoice_no : 98.998998998999
total_amount: 96.7967967967968
contact: 98.8988988988989
Overall: 96.69669669669669


,template,filename,contact_tru,contact_hat,total_tru,total_hat,invoice_tru,invoice_hat,invoice_matched,total_hat_amount,total_matched,contact_score,contact_matched,document_matched
0,invoice_template_01,001,"960 Hurley Springs North Alyssa, RI 49322\nHay...",Hayes Mercedes LLC Martinez\n960 Hurley Spring...,534.11,£534.11,802205,802205,True,534.11,True,100.000000,True,True
1,invoice_template_01,002,Unit 1426 Box 5236 DPO AE 74575\nWilson PLC\nA...,Wilson PLC\nDiane Myers\nUnit 1426 Box 5236\nD...,756.81,£756.81,706585,706585,True,756.81,True,100.000000,True,True
2,invoice_template_01,003,"219 Hall Plaza Emilymouth, TN 77611\nMcbride I...",Mcbride Inc\nMelissa Miller\n219 Hall Plaza\nE...,350.83,£350.83,436251,436251,True,350.83,True,100.000000,True,True
3,invoice_template_01,004,"44126 Claudia Mount Suite 784 Morrisfort, IA 9...","Barnett, Nicole 44126 Lao White Martin Claudia...",704.26,£704.26,807426,807426,True,704.26,True,96.170213,True,True
4,invoice_template_01,005,"19780 Miller Rest Apt. 044 West Kristenfurt, M...",Sanchez Inc\nRyan Davis\n19780 Miller Rest Apt...,699.63,£699.63,614625,614625,True,699.63,True,100.000000,True,True
